# CReM baseline

This notebook applies CReM to the same fixed parent molecules used in the benchmark.

For each parent molecule, CReM enumerates local mutations using the ChEMBL-derived fragment database. Generation is **descriptor-agnostic**: HBD, HBA, aromatic rings, and rotatable bonds are not used to guide mutation. Descriptor changes are evaluated only after generation.


## Benchmark conditions

- Parent set: `data/chembl_1000_parents.csv`
- Maximum retained products per parent: `N_CANDIDATES_PER_PARENT`
- CReM database: `chembl33_sa2_f5.db`
- If CReM produces more than the allowed number of unique products, a fixed-seed random sample is retained.
- Invalid structures, unchanged parents, and duplicate canonical SMILES are excluded.

The same descriptor calculation function is used as in the Random substitution benchmark.


## 1. Imports and benchmark settings


In [1]:
import random
import sys
from pathlib import Path

import pandas as pd
from tqdm import tqdm
from rdkit import Chem
from crem.crem import mutate_mol

sys.path.append("..")

from configs.benchmark_config import (
    N_CANDIDATES_PER_PARENT,
    RANDOM_SEED,
)
from utils.parents import prepare_parent
from utils.records import append_candidate_record

random.seed(RANDOM_SEED)

CREM_DB = Path.home() / "software/CReM_data/chembl33_sa2_f5.db"

## 2. Load the fixed parent compounds

The same parent CSV is used for Random substitution, CReM, scaffold-tuner, and REINVENT4 Mol2Mol.


In [2]:
df = pd.read_csv("../data/chembl_1000_parents.csv")

print(f"Number of parent compounds: {len(df)}")
df.head()

Number of parent compounds: 1000


,chembl_id,smiles,mw,hbd,hba,logp,rotb,ar
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,367.449,2,4,4.49360,9,2
1,CHEMBL91137,CCCc1nnc([S+]([O-])Cc2ncc(C)c(OC)c2C)o1,309.391,0,6,2.35034,6,2
2,CHEMBL441229,C=C1C(O)C(O)C(O)C(O)C1O,176.168,5,5,-2.63930,0,0
3,CHEMBL1068,NC(=O)N1c2ccccc2CC(=O)c2ccccc21,252.273,1,2,2.64220,0,2
4,CHEMBL105373,COc1cc(-n2sc3ncccc3c2=O)cc(OC)c1OC,318.354,0,6,2.47300,4,3


## 3. Generate CReM products and calculate descriptor changes

CReM mutations are generated independently of the target descriptor.  
All returned structures are converted to canonical isomeric SMILES and deduplicated.

If more than `N_CANDIDATES_PER_PARENT` unique products are available, a fixed-seed random subset is retained so that the maximum number of evaluated products per parent is the same as in the other benchmark methods.


In [3]:
records = []

for _, row in tqdm(df.iterrows(), total=len(df), desc="CReM"):
    prepared = prepare_parent(row)
    if prepared is None:
        continue

    chembl_id, parent_smiles, parent_desc = prepared
    parent = Chem.MolFromSmiles(parent_smiles)

    generated_smiles = set()

    try:
        for candidate_smiles in mutate_mol(
            parent,
            db_name=str(CREM_DB),
            max_size=1,
        ):
            new_mol = Chem.MolFromSmiles(candidate_smiles)
            if new_mol is None:
                continue

            new_smiles = Chem.MolToSmiles(
                new_mol,
                canonical=True,
                isomericSmiles=True,
            )

            if (
                new_smiles == parent_smiles
                or new_smiles in generated_smiles
            ):
                continue

            generated_smiles.add(new_smiles)

            if len(generated_smiles) >= N_CANDIDATES_PER_PARENT:
                break

    except Exception:
        continue

    for new_smiles in generated_smiles:
        append_candidate_record(
            records=records,
            chembl_id=chembl_id,
            parent_smiles=parent_smiles,
            generated_smiles=new_smiles,
            parent_desc=parent_desc,
        )

df_crem = pd.DataFrame(records)

print(df_crem.shape)
df_crem.head()

CReM:   0%|          | 0/1000 [00:00<?, ?it/s]

CReM: 100%|██████████| 1000/1000 [00:38<00:00, 26.28it/s]


(24992, 15)


,chembl_id,parent_smiles,generated_smiles,hbd_parent,hbd_generated,delta_hbd,hba_parent,hba_generated,delta_hba,ar_parent,ar_generated,delta_ar,rotb_parent,rotb_generated,delta_rotb
0,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,CCCOC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc...,2,2,0,4,4,0,2,2,0,9,11,2
1,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,CCOC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2...,2,2,0,4,4,0,2,2,0,9,10,1
2,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,2,2,0,4,4,0,2,2,0,9,10,1
3,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,COC(=O)CCN(C)CCC1=NC(=Cc2[nH]c(-c3ccc[nH]3)cc2...,2,2,0,4,5,1,2,2,0,9,9,0
4,CHEMBL85118,COC(=O)CCCCCC1=N/C(=C\c2[nH]c(-c3ccc[nH]3)cc2O...,CCCOC(=O)CCCCCC1=NC(=Cc2[nH]c(-c3ccc[nH]3)cc2O...,2,2,0,4,4,0,2,2,0,9,11,2


## 4. Save generated products

The output columns intentionally match the Random substitution output so that all methods can be evaluated with the same downstream analysis code.

In [4]:
OUTPUT_FILE = "../results/crem.csv"

df_crem.to_csv(OUTPUT_FILE, index=False)
print(f"Saved: {OUTPUT_FILE}")

Saved: ../results/crem.csv


## Output and downstream evaluation

- `results/crem.csv` — product-level CReM results.

The common analysis notebook should evaluate the same criteria for every method:

- HBD increase/decrease: `delta_hbd == ±1`
- HBA increase/decrease: `delta_hba == ±1`
- Aromatic-ring increase/decrease: `delta_ar == ±1`
- Rotatable-bond increase/decrease: `delta_rotb == ±1`
- Selective HBD increase: `delta_hbd == +1` and `delta_hba == 0`

For the final paper release, record the CReM package version, database file/version, mutation parameters, fixed parent set, and Git commit/release tag.
